# Explore or exploit: bandits
**Machine Learning · Week 7 (Reinforcement learning) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w7d2.html

Four cookie jars. Each one gives a delicious cookie with some hidden probability. You may open one
jar per turn and want as many good cookies as possible. That's a **multi-armed bandit**, the simplest
reinforcement-learning problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

true_chance = np.array([0.30, 0.55, 0.45, 0.70])     # hidden from the agent!

def open_jar(jar, rng):
    return float(rng.random() < true_chance[jar])     # reward: 1 for a good cookie, 0 for a burnt one

def epsilon_greedy(epsilon, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    estimate = np.zeros(4)          # our guess of each jar's chance
    count = np.zeros(4)             # how often we opened each jar
    rewards = []
    for t in range(steps):
        if rng.random() < epsilon:
            jar = int(rng.integers(4))              # explore: try a random jar
        else:
            jar = int(np.argmax(estimate))          # exploit: open the best jar so far
        r = open_jar(jar, rng)
        count[jar] += 1
        estimate[jar] += (r - estimate[jar]) / count[jar]   # running average
        rewards.append(r)
    return np.array(rewards), estimate, count

rewards, estimate, count = epsilon_greedy(epsilon=0.1)
print("estimated chances:", estimate.round(2), "   true chances:", true_chance)
print("times each jar was opened:", count.astype(int))
print("average reward:", rewards.mean().round(3))

## How much should we explore?
One run is noisy, so we average 200 runs for each ε.

In [ ]:
steps = np.arange(1, 1001)
for eps in [0.0, 0.1, 0.3]:
    curves = np.array([np.cumsum(epsilon_greedy(eps, 1000, seed)[0]) / steps for seed in range(200)])
    plt.plot(steps, curves.mean(axis=0), label=f"ε = {eps}")
plt.axhline(true_chance.max(), ls="--", color="#8a85a3", label="best jar every time")
plt.xlabel("step")
plt.ylabel("average reward so far")
plt.ylim(0.3, 0.75)
plt.legend()
plt.title("Explore a little, exploit a lot")
plt.show()

With ε = 0 (never explore) the agent opens jar 0 first and never tries anything else: the other estimates
stay at 0, so jar 0 always looks best. With ε = 0.3 it learns fastest at first, but keeps wasting 30% of its
turns on random jars forever. A small ε wins in the long run.

## Regret
**Regret** is the reward you missed compared with always opening the best jar.

In [ ]:
best_total = 1000 * true_chance.max()
for eps in [0.0, 0.1, 0.3]:
    total = np.mean([epsilon_greedy(eps, 1000, seed)[0].sum() for seed in range(200)])
    print(f"ε = {eps}: average cookies {total:5.1f}   regret {best_total - total:5.1f}")

## Your turn: UCB, optimism with a reason
**UCB** (upper confidence bound) explores where it is still *unsure*. Open each jar once, then always open
the jar with the highest $Q(a) + c\sqrt{\ln t / n_a}$: the estimate plus a bonus that shrinks as the jar
is opened more often ($n_a$ times so far, $t$ turns in total). Write `ucb(c, steps, seed)` like
`epsilon_greedy`, and compare its average regret over 200 runs with ε-greedy (ε = 0.1), for $c = 0.5$ and $c = 2$.

In [ ]:
def ucb(c, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    estimate, count, rewards = np.zeros(4), np.zeros(4), []
    for t in range(steps):
        jar = t % 4    # TODO: open each jar once (t < 4), then argmax of estimate + c * sqrt(log(t) / count)
        r = open_jar(jar, rng)
        count[jar] += 1
        estimate[jar] += (r - estimate[jar]) / count[jar]
        rewards.append(r)
    return np.array(rewards), estimate, count

for name, run in [("eps-greedy 0.1", lambda s: epsilon_greedy(0.1, 1000, s)), ("UCB c = 0.5", lambda s: ucb(0.5, 1000, s))]:
    print(name, "regret", round(best_total - np.mean([run(s)[0].sum() for s in range(200)]), 1))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def ucb(c, steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    estimate, count, rewards = np.zeros(4), np.zeros(4), []
    for t in range(steps):
        if t < 4:
            jar = t                                              # try every jar once
        else:
            jar = int(np.argmax(estimate + c * np.sqrt(np.log(t) / count)))
        r = open_jar(jar, rng)
        count[jar] += 1
        estimate[jar] += (r - estimate[jar]) / count[jar]
        rewards.append(r)
    return np.array(rewards), estimate, count

for name, run in [("eps-greedy, eps = 0.1", lambda s: epsilon_greedy(0.1, 1000, s)),
                  ("UCB, c = 0.5", lambda s: ucb(0.5, 1000, s)),
                  ("UCB, c = 2", lambda s: ucb(2.0, 1000, s))]:
    print(f"{name:22s} average regret {best_total - np.mean([run(s)[0].sum() for s in range(200)]):5.1f}")
print("A modest bonus explores only where it's still unsure; too big a bonus keeps exploring for too long.")